# Model Optimization: WANDA Pruning

In this notebook, we'll apply WANDA (Weight ANalysis for Deep leArning) pruning techniques to our models using distributed processing. This is an advanced pruning technique that considers both weight magnitudes and activation statistics when deciding which weights to prune.

## What is WANDA Pruning?

WANDA pruning is an advanced technique that improves upon traditional magnitude-based pruning methods by incorporating activation statistics. While standard pruning methods like L1 unstructured pruning only look at the absolute values of weights, WANDA considers how those weights interact with activations during inference.

### Key Differences from Standard Pruning:
- **Activation-Aware**: Considers both weight magnitudes and activation statistics
- **Better Accuracy Preservation**: Tends to maintain model accuracy better than simple magnitude pruning
- **Calibration Data**: Uses a small set of sample inputs to collect activation statistics
- **Importance Scoring**: Calculates importance as weight magnitude × activation magnitude

## 1. Import Dependencies

In [ ]:
import json
import time
import pandas as pd
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
import time
from IPython.display import clear_output

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")

## 3. Load Model Information and Previous Pruning Results

In [ ]:
# Try to load model information from file
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }

# Try to load standard pruned metrics for comparison
try:
    with open('pruned-metrics.json', 'r') as f:
        pruned_metrics = json.load(f)
    print(f"Loaded standard pruned metrics for {len(pruned_metrics)} models")
except FileNotFoundError:
    print("pruned-metrics.json not found. Will proceed without standard pruning metrics for comparison.")
    pruned_metrics = {}

## 4. Model Selection for WANDA Pruning

Before configuring our WANDA pruning jobs, we need to carefully select which models are suitable for pruning. Based on extensive research and experimentation, we've found that not all model architectures respond well to pruning techniques.

### Models Not Suitable for Pruning

**Named Entity Recognition (NER) Models**: Token classification models like BERT-based NER are highly sensitive to pruning due to:

1. **Token-level Classification Sensitivity**: NER models make token-by-token predictions that rely heavily on contextual relationships between tokens
2. **Attention Mechanism Importance**: The attention mechanisms in transformer models are critical for capturing token relationships, and pruning disrupts these mechanisms
3. **Entity Type Sensitivity**: Different entity types (Person, Organization, Location) show varying levels of sensitivity to pruning
4. **Boundary Detection Issues**: Even with minimal pruning (3%), entity boundary detection is significantly affected

For these models, we recommend alternative optimization approaches like quantization or knowledge distillation instead.

### Filtering Models for WANDA Pruning

We'll filter our model list to exclude NER/token-classification models before proceeding with pruning:

In [ ]:
# Filter out models that are not suitable for pruning
prunable_models = {}
excluded_models = {}

for model_key, info in model_info.items():
    if info['task'] == 'token-classification':
        excluded_models[model_key] = info
        print(f"Excluding {model_key} ({info['model_name']}) from WANDA pruning as token-classification models are not suitable for pruning")
    else:
        prunable_models[model_key] = info
        print(f"Including {model_key} ({info['model_name']}) for WANDA pruning")

print(f"\nSelected {len(prunable_models)} models for WANDA pruning out of {len(model_info)} total models")

# Check if we have any models to prune
if len(prunable_models) == 0:
    print("\n⚠️ No suitable models found for WANDA pruning. Please add models with supported tasks.")
    print("Supported tasks include: text-classification, question-answering, etc.")
    print("Token-classification (NER) models are not recommended for pruning.")

## 5. Launch Distributed WANDA Pruning Jobs

Now that we've filtered our models to include only those suitable for pruning, we'll set up and launch the SageMaker Processing jobs to perform WANDA pruning. Each suitable model will be processed in a separate job, allowing for parallel processing.

In [ ]:
# Define the instance type to use for pruning
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="2.0.0",
    py_version="py310",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="wanda-pruning",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch WANDA pruning jobs for suitable models in parallel
job_names = []  # List to store all job names
job_output_paths = {}
s3_client = boto3.client('s3')

# First, prepare all the job configurations
job_configs = {}
print("Preparing WANDA pruning jobs for suitable models...")

for model_key in prunable_models.keys():
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: prunable_models[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}-wanda-pruned'
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='wanda-pruned-model',
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Store the job configuration
    job_configs[model_key] = {
        'inputs': inputs,
        'outputs': outputs,
        'arguments': [
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--pruning-amount', '0.3',
            '--calibration-samples', '32'  # Number of samples to use for activation statistics
        ],
        'output_path': output_path  # Store the output path in the job config for easy access later
    }
    print(f"Prepared job configuration for {model_key}")

# Now launch all jobs in parallel
print("\nLaunching WANDA pruning jobs in parallel...")

# Check if we have any models to prune
if len(job_configs) == 0:
    print("No suitable models to prune. Skipping job launch.")
else:
    for model_key, config in job_configs.items():
        try:
            # Create a unique job name with timestamp to avoid conflicts
            timestamp = int(time.time())
            job_name = f"wanda-pruning-{model_key}-{timestamp}"
            
            # Run the processing job with the unique name
            processor.run(
                code='wanda_pruning_script.py',
                source_dir='wanda_pruning_scripts',
                inputs=config['inputs'],
                outputs=config['outputs'],
                arguments=config['arguments'],
                wait=False,  # Don't wait for the job to complete before continuing
                job_name=job_name  # Explicitly set the job name
            )
            
            # Store the job name for tracking
            job_names.append(job_name)
            print(f"Launched job for {model_key}: {job_name}")
        except Exception as e:
            print(f"Error launching job for {model_key}: {e}")

    print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

In [ ]:
# Monitor job status
from sagemaker.processing import ProcessingJob

# Function to check if all jobs are complete
def are_all_jobs_complete(job_names, sagemaker_session):
    all_complete = True
    job_statuses = {}
    
    # Create a SageMaker client for API calls
    sagemaker_client = boto3.client('sagemaker')
    
    for job_name in job_names:
        try:
            # Use the SageMaker client to describe the processing job
            response = sagemaker_client.describe_processing_job(
                ProcessingJobName=job_name
            )
            status = response['ProcessingJobStatus']
            job_statuses[job_name] = status
            
            if status in ['InProgress', 'Stopping']:
                all_complete = False
        except Exception as e:
            job_statuses[job_name] = f"Error: {str(e)}"
            # Consider jobs with errors as complete to avoid infinite loops
            
    return all_complete, job_statuses

# Poll for job completion if there are any jobs running
if len(job_names) > 0:
    print("Waiting for all jobs to complete...")
    while True:
        all_complete, job_statuses = are_all_jobs_complete(job_names, sagemaker_session)
        
        # Clear previous output
        clear_output(wait=True)
        
        # Print current status
        print("Current job statuses:")
        for job_name, status in job_statuses.items():
            print(f"Job {job_name}: {status}")
        
        if all_complete:
            print("All jobs completed!")
            break
        
        print("Waiting for jobs to complete... Will check again in 60 seconds.")
        time.sleep(60)  # Check every minute

    print("\nAll jobs have completed or failed.")
else:
    print("No WANDA pruning jobs were launched. Skipping job monitoring.")

## 5. Analyze Pruned Models

Now that the WANDA pruning jobs are complete, we'll analyze the pruned models by comparing them to the original models. We'll measure:

1. **Size Reduction**: How much smaller the pruned model is
2. **Parameter Reduction**: How many parameters were removed
3. **Sparsity**: The percentage of zero weights in the model
4. **Inference Time Improvement**: How much faster the pruned model is
5. **Memory Usage Reduction**: How much less memory the pruned model uses

First, let's define the functions we'll use to collect metrics:

In [ ]:
# Define functions for metrics collection
def get_model_size(model):
    """Calculate model size in MB."""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

def get_num_parameters(model):
    """Calculate number of parameters in the model."""
    return sum(p.numel() for p in model.parameters())

def count_non_zero_params(model):
    """Count non-zero parameters in the model."""
    non_zero = 0
    total = 0
    for param in model.parameters():
        if param.dim() > 1:  # Only count weights, not biases
            non_zero += torch.count_nonzero(param).item()
            total += param.numel()
    return non_zero, total

def measure_inference_time(model, inputs, num_runs=10):
    """Measure average inference time over multiple runs."""
    # Warm-up run
    with torch.no_grad():
        model(**inputs)
    
    # Measure inference time
    start_event = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None
    end_event = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None
    
    inference_times = []
    for _ in range(num_runs):
        if torch.cuda.is_available():
            start_event.record()
            with torch.no_grad():
                model(**inputs)
            end_event.record()
            torch.cuda.synchronize()
            inference_times.append(start_event.elapsed_time(end_event))
        else:
            start_time = time.time()
            with torch.no_grad():
                model(**inputs)
            end_time = time.time()
            inference_times.append((end_time - start_time) * 1000)  # Convert to ms
    
    return sum(inference_times) / len(inference_times)

def measure_memory_usage(model, inputs):
    """Measure peak memory usage during inference."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        
        with torch.no_grad():
            model(**inputs)
        
        memory_usage = torch.cuda.max_memory_allocated() / 1024**2  # Convert to MB
    else:
        # For CPU, use a rough estimate based on model size
        memory_usage = get_model_size(model) * 2  # Rough estimate
    
    return memory_usage

def prepare_sample_inputs(model_name, task, tokenizer, device):
    """Prepare sample inputs for the model based on its task."""
    if task == "sequence-classification" or task == "text-classification":
        text = "I really enjoyed this movie. The acting was superb and the plot was engaging."
        inputs = tokenizer(text, return_tensors="pt")
    elif task == "token-classification":
        text = "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington."
        inputs = tokenizer(text, return_tensors="pt")
    elif task == "question-answering":
        question = "What is machine learning?"
        context = "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
        inputs = tokenizer(question, context, return_tensors="pt")
    elif task == "masked-lm" or task == "fill-mask":
        text = "The [MASK] is a large language model trained by OpenAI."
        inputs = tokenizer(text, return_tensors="pt")
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    # Move inputs to the appropriate device
    return {k: v.to(device) for k, v in inputs.items()}

In [ ]:
# Collect metrics for all pruned models
wanda_pruned_metrics = {}

# Check if we have any pruned models to analyze
if len(job_configs) == 0:
    print("No WANDA pruned models to analyze. This could be because:")
    print("1. No suitable models were found for pruning (e.g., only NER models were available)")
    print("2. The pruning jobs failed to complete successfully")
    print("\nTo analyze WANDA pruned models, please add models with supported tasks like text-classification.")
else:
    # Import necessary libraries for model loading and evaluation
    import torch
    import os
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from transformers import AutoModelForTokenClassification, AutoModelForQuestionAnswering
    from transformers import AutoModelForMaskedLM

    for model_key, job_info in job_configs.items():
        print(f"\nAnalyzing WANDA pruned model: {model_key}")
        
        # Get model info
        model_info_item = prunable_models[model_key]
        model_name = model_info_item["model_name"]
        task = model_info_item["task"]
        
        # Set device
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")
        
        # Load original model
        print(f"Loading original model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        if task == "sequence-classification" or task == "text-classification":
            original_model = AutoModelForSequenceClassification.from_pretrained(model_name)
        elif task == "token-classification":
            original_model = AutoModelForTokenClassification.from_pretrained(model_name)
        elif task == "question-answering":
            original_model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        elif task == "masked-lm" or task == "fill-mask":
            original_model = AutoModelForMaskedLM.from_pretrained(model_name)
        else:
            raise ValueError(f"Unsupported task: {task}")
        
        original_model = original_model.to(device)
        original_model.eval()
        
        # Prepare sample inputs
        inputs = prepare_sample_inputs(model_name, task, tokenizer, device)
        
        # Measure baseline metrics
        baseline_size = get_model_size(original_model)
        baseline_params = get_num_parameters(original_model)
        baseline_non_zero, baseline_total = count_non_zero_params(original_model)
        baseline_inference_time = measure_inference_time(original_model, inputs)
        baseline_memory_usage = measure_memory_usage(original_model, inputs)
        
        print(f"Original model size: {baseline_size:.2f} MB")
        print(f"Original parameters: {baseline_params:,}")
        print(f"Original non-zero weights: {baseline_non_zero:,}/{baseline_total:,} ({baseline_non_zero/baseline_total*100:.2f}%)")
        print(f"Original inference time: {baseline_inference_time:.2f} ms")
        print(f"Original memory usage: {baseline_memory_usage:.2f} MB")
        
        # Load pruned model
        pruned_model_path = os.path.join(job_info["output_path"], f"{model_key}_pruned")
        print(f"Loading pruned model from: {pruned_model_path}")
        
        try:
            if task == "sequence-classification" or task == "text-classification":
                pruned_model = AutoModelForSequenceClassification.from_pretrained(pruned_model_path)
            elif task == "token-classification":
                pruned_model = AutoModelForTokenClassification.from_pretrained(pruned_model_path)
            elif task == "question-answering":
                pruned_model = AutoModelForQuestionAnswering.from_pretrained(pruned_model_path)
            elif task == "masked-lm" or task == "fill-mask":
                pruned_model = AutoModelForMaskedLM.from_pretrained(pruned_model_path)
            
            pruned_model = pruned_model.to(device)
            pruned_model.eval()
            
            # Measure pruned metrics
            pruned_size = get_model_size(pruned_model)
            pruned_params = get_num_parameters(pruned_model)
            pruned_non_zero, pruned_total = count_non_zero_params(pruned_model)
            pruned_inference_time = measure_inference_time(pruned_model, inputs)
            pruned_memory_usage = measure_memory_usage(pruned_model, inputs)
            
            print(f"Pruned model size: {pruned_size:.2f} MB")
            print(f"Pruned parameters: {pruned_params:,}")
            print(f"Pruned non-zero weights: {pruned_non_zero:,}/{pruned_total:,} ({pruned_non_zero/pruned_total*100:.2f}%)")
            print(f"Pruned inference time: {pruned_inference_time:.2f} ms")
            print(f"Pruned memory usage: {pruned_memory_usage:.2f} MB")
            
            # Calculate improvements
            size_reduction = (baseline_size - pruned_size) / baseline_size * 100
            param_reduction = (baseline_params - pruned_params) / baseline_params * 100
            sparsity = (1 - pruned_non_zero / pruned_total) * 100
            time_improvement = (baseline_inference_time - pruned_inference_time) / baseline_inference_time * 100
            memory_reduction = (baseline_memory_usage - pruned_memory_usage) / baseline_memory_usage * 100
            
            print(f"Size reduction: {size_reduction:.2f}%")
            print(f"Parameter reduction: {param_reduction:.2f}%")
            print(f"Model sparsity: {sparsity:.2f}%")
            print(f"Inference time improvement: {time_improvement:.2f}%")
            print(f"Memory usage reduction: {memory_reduction:.2f}%")
            
            # Save metrics
            wanda_pruned_metrics[model_key] = {
                "model_name": model_name,
                "task": task,
                "pruning_method": "wanda",
                "pruning_amount": 0.3,
                "model_size": round(pruned_size, 2),
                "inference_time": round(pruned_inference_time, 2),
                "memory_usage": round(pruned_memory_usage, 2),
                "size_reduction": round(size_reduction, 2),
                "time_improvement": round(time_improvement, 2),
                "memory_reduction": round(memory_reduction, 2),
                "parameter_reduction": round(param_reduction, 2),
                "sparsity": round(sparsity, 2),
                "non_zero_weights": pruned_non_zero,
                "total_weights": pruned_total
            }
            
        except Exception as e:
            print(f"Error analyzing pruned model {model_key}: {e}")
            import traceback
            traceback.print_exc()

    # Save all metrics to a file if we have any
    if wanda_pruned_metrics:
        metrics_path = "wanda-pruned-metrics.json"
        with open(metrics_path, "w") as f:
            json.dump(wanda_pruned_metrics, f, indent=2)
        print(f"\nSaved WANDA pruned metrics to {metrics_path}")
    else:
        print("\nNo metrics to save as no pruned models were successfully analyzed.")

## 6. Compare WANDA Pruning with Standard Pruning

Now we'll compare the results of WANDA pruning with standard pruning to see the differences in model size, inference time, and accuracy.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

# Function to get total size of objects with a prefix from S3
def get_total_size(bucket, prefix):
    total_size = 0
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                total_size += obj['Size']
    return total_size

# Check if we have any metrics to compare
if len(wanda_pruned_metrics) == 0:
    print("No WANDA pruned models available for comparison.")
    print("This is expected if you only have NER/token-classification models in your model_info.json file.")
    print("To see comparison results, add models with supported tasks like text-classification.")
else:
    for model_key in wanda_pruned_metrics.keys():
        standard_pruned = pruned_metrics.get(model_key, {})
        wanda_pruned = wanda_pruned_metrics[model_key]
        
        # Get original model size from S3
        if model_key in prunable_models and 's3_uri' in prunable_models[model_key]:
            original_model_prefix = prunable_models[model_key]['s3_uri'].replace(f"s3://{S3_BUCKET}/", "")
            original_size = get_total_size(S3_BUCKET, original_model_prefix)
            original_size_mb = original_size / (1024 * 1024)
        else:
            # If we can't find the S3 URI, use a default value
            print(f"Warning: Could not find S3 URI for model {model_key}. Using estimated size.")
            original_size_mb = wanda_pruned.get('model_size', 0) / (1 - wanda_pruned.get('size_reduction', 0)/100)
        
        # Get WANDA pruned model size
        wanda_model_size_mb = wanda_pruned.get('model_size', 0)
        
        # Calculate size reduction percentage
        wanda_size_reduction = ((original_size_mb - wanda_model_size_mb) / original_size_mb) * 100 if original_size_mb > 0 else 0
        
        # Prepare data for this model
        model_data = {
            'Model': wanda_pruned['model_name'],
            'Original Size (MB)': f"{original_size_mb:.2f}",
            'WANDA Pruning Amount': f"{wanda_pruned['pruning_amount'] * 100:.1f}%",
            'WANDA Pruned Size (MB)': f"{wanda_model_size_mb:.2f}",
            'WANDA Size Reduction (%)': f"{wanda_size_reduction:.2f}",
            'WANDA Sparsity (%)': f"{wanda_pruned.get('sparsity', 0):.2f}"
        }
        
        # Add standard pruning data if available
        if standard_pruned:
            standard_model_size_mb = standard_pruned.get('model_size', 0)
            standard_size_reduction = ((original_size_mb - standard_model_size_mb) / original_size_mb) * 100 if original_size_mb > 0 else 0
            
            model_data.update({
                'Standard Pruning Method': standard_pruned.get('pruning_method', 'N/A'),
                'Standard Pruning Amount': f"{standard_pruned.get('pruning_amount', 0) * 100:.1f}%",
                'Standard Pruned Size (MB)': f"{standard_model_size_mb:.2f}",
                'Standard Size Reduction (%)': f"{standard_size_reduction:.2f}",
                'Standard Sparsity (%)': f"{standard_pruned.get('sparsity', 0):.2f}"
            })
        
        comparison_data.append(model_data)

    # Create DataFrame
    comparison_df = pd.DataFrame(comparison_data)

    # Display the DataFrame
    display(comparison_df)

## 7. Next Steps

Now that we've applied WANDA pruning to our models and compared it with standard pruning, we'll explore knowledge distillation in the next notebook to create even smaller, more efficient models.

### What We've Learned:
- How WANDA pruning differs from standard pruning by considering activation statistics
- How to implement WANDA pruning using SageMaker Processing jobs
- How WANDA pruning compares to standard pruning in terms of model size reduction and sparsity
- How to run multiple processing jobs in parallel for faster experimentation

### What's Next - Knowledge Distillation:
Knowledge distillation is a technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model. This approach can lead to even more significant size reductions while maintaining good performance.